# Train the Student — Gemma-3-270M on Hop-1 Decomposition

Notebook 02 labeled HotpotQA with GPT-4o and formatted the rows into chat-template `messages`. Notebook 03 built the scoring harness. Here I fine-tune Gemma-3-270M on the teacher's Hop-1 decompositions and generate first-step decompositions for the 95 held-out test questions.

This notebook trains one ablation variant at a time. It runs on a Colab GPU (T4 or L4). The output is `preds_{variant}.jsonl`, which I download and score locally through notebook 03.

Variant this run: **sys** (a short fixed `system` turn prepended to `user`/`assistant`).

## 1. Environment and GPU

In [1]:
!pip install -q -U transformers trl datasets accelerate huggingface_hub

import torch

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

CUDA: True
GPU : Tesla T4


Gemma-3-270M is a gated model on the Hub, so I authenticate with a read token (accept the license on the model page first). The token lives only in this session.

In [2]:
# import getpass
# from huggingface_hub import login

# login(getpass.getpass("Hugging Face token: "))

from google.colab import userdata
hf_token = userdata.get("HF_TOKEN")

## 2. Load the training data

I upload the two sys files from `data/`: `train_sys.jsonl` and `test_sys.jsonl`. Training only needs the `messages` column; the extra fields (`gold_titles`, `type`, and so on) are kept in the test file for scoring later.

In [3]:
from google.colab import files

# Upload train_sys.jsonl and test_sys.jsonl when prompted.
uploaded = files.upload()

Saving test_sys.jsonl to test_sys.jsonl
Saving train_sys.jsonl to train_sys.jsonl


In [4]:
from datasets import load_dataset

VARIANT = "sys"

# SFTTrainer reads the conversational `messages` column and applies the chat
# template itself, so I drop every other column from the training set.
train_ds = load_dataset(
    "json", data_files=f"train_{VARIANT}.jsonl", split="train"
).select_columns(["messages"])

print(train_ds)
print(train_ds[0]["messages"])

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages'],
    num_rows: 379
})
[{'role': 'system', 'content': 'You are a reasoning agent. Given a multi-hop question, output only the first reasoning step as YAML with three fields: thought, action, target_entity.'}, {'role': 'user', 'content': 'Pearl Lowe and Alison Goldfrapp, is of which nationality?'}, {'role': 'assistant', 'content': 'thought: "I need to find the nationality of Pearl Lowe first."\naction: "Lookup"\ntarget_entity: "Pearl Lowe"'}]


## 3. Load the base model and tokenizer

In [ ]:
# from transformers import pipeline

# pipe = pipeline("text-generation", model="google/gemma-3-270m-it")
# message = [
#     {"role": "user", "content": "Who are you?"},
# ]
# pipe(message)

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "google/gemma-3-270m-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# Pin fp32 explicitly: recent transformers defaults from_pretrained to the
# checkpoint's own dtype, and Gemma-3's checkpoint is bf16. T4 has no bf16
# tensor cores, so that silent default broke the fp16 GradScaler below.
# fp32 weights + fp16=True in SFTConfig is the standard T4 mixed-precision
# setup: master weights stay fp32, autocast handles the fp16 compute.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    attn_implementation="eager",
    torch_dtype=torch.float32)

print(model.config.model_type, f"{model.num_parameters()/1e6:.0f}M params")

config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  536MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

gemma3_text 268M params


## 4. Training configuration

The knobs that decide what the student actually learns: how many epochs, the learning rate, the batch size, the sequence length, the precision, and whether the loss is computed on the whole sequence or only on the assistant turn.

This cell must define `sft_config` (an `SFTConfig`).

In [6]:
from trl import SFTConfig

sft_config = SFTConfig(
    output_dir="gemma3-270m-hop1-sys",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    learning_rate=5e-5,
    max_length=256,
    fp16=True,
    assistant_only_loss=True,
    logging_steps=10,
    report_to="none",
)


## 5. Fine-tune

In [7]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    processing_class=tokenizer,
)

trainer.train()

Tokenizing train dataset:   0%|          | 0/379 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/379 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/379 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/379 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
10,1.560403
20,0.440201
30,0.349052
40,0.366379
50,0.252770
60,0.189469
70,0.138228
80,0.148888
90,0.153470
100,0.114351


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=144, training_loss=0.2751341540780332, metrics={'train_runtime': 92.829, 'train_samples_per_second': 12.248, 'train_steps_per_second': 1.551, 'total_flos': 94106806268160.0, 'train_loss': 0.2751341540780332, 'entropy': 0.06464770808815956, 'num_tokens': 123897.0, 'mean_token_accuracy': 0.9835876971483231, 'epoch': 3.0})

In [8]:
## save the model and tokenizer
trainer.save_model("gemma3-270m-hop1-sys")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [19]:
!ls ./gemma3-270m-hop1-sys/

chat_template.jinja  generation_config.json  tokenizer_config.json
checkpoint-144	     model.safetensors	     tokenizer.json
config.json	     README.md		     training_args.bin


In [11]:
!zip -r gemma3-270m-hop1-sys.zip ./gemma3-270m-hop1-sys/
files.download("gemma3-270m-hop1-sys.zip")

updating: gemma3-270m-hop1-sys/ (stored 0%)
updating: gemma3-270m-hop1-sys/README.md (deflated 43%)
updating: gemma3-270m-hop1-sys/training_args.bin (deflated 53%)
updating: gemma3-270m-hop1-sys/config.json (deflated 67%)
updating: gemma3-270m-hop1-sys/model.safetensors (deflated 8%)
updating: gemma3-270m-hop1-sys/chat_template.jinja (deflated 70%)
updating: gemma3-270m-hop1-sys/tokenizer.json (deflated 83%)
updating: gemma3-270m-hop1-sys/tokenizer_config.json (deflated 60%)
updating: gemma3-270m-hop1-sys/checkpoint-144/ (stored 0%)
updating: gemma3-270m-hop1-sys/checkpoint-144/training_args.bin (deflated 53%)
updating: gemma3-270m-hop1-sys/checkpoint-144/config.json (deflated 67%)
updating: gemma3-270m-hop1-sys/checkpoint-144/rng_state.pth (deflated 27%)
updating: gemma3-270m-hop1-sys/checkpoint-144/model.safetensors (deflated 8%)
updating: gemma3-270m-hop1-sys/checkpoint-144/chat_template.jinja (deflated 70%)
updating: gemma3-270m-hop1-sys/checkpoint-144/tokenizer.json (deflated 83%)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Generate Hop-1 decompositions for the test set

Inference has to match training exactly: I apply the same chat template, but stop after the prompt and let the model complete the assistant turn. The gold assistant message is dropped so the model has to produce it.

This cell must define `generate_hop1(messages) -> str`, where `messages` is the full test row's `messages` list (with the gold assistant turn) and the return value is the model's raw YAML string.

In [18]:
# Define `generate_hop1(messages) -> str`.
def generate_hop1(messages)->str:
  inputs = tokenizer.apply_chat_template(
      messages[:-1],
      tokenize=True,
      add_generation_prompt=True,
      return_tensors="pt"
  )

  print(type(inputs))

  inputs = inputs.to(model.device)

  outputs = model.generate(
      **inputs,
      max_new_tokens=150,
      do_sample=False
  )

  input_length = inputs['input_ids'].shape[1]
  generated_tokens = outputs[0][input_length:]

  return tokenizer.decode(generated_tokens, skip_special_tokens=True)


In [ ]:
model.eval()

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 640, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=640, out_features=1024, bias=False)
          (k_proj): Linear(in_features=640, out_features=256, bias=False)
          (v_proj): Linear(in_features=640, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=640, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
          (up_proj): Linear(in_features=640, out_features=2048, bias=False)
          (down_proj): Linear(in_features=2048, out_features=640, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)

In [ ]:
test_rows = load_dataset(
    'json', data_files='test_sys.jsonl', split='train'
).select_columns(['messages'])

print(test_rows)
print(test_rows[0]['messages'])

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages'],
    num_rows: 95
})
[{'role': 'system', 'content': 'You are a reasoning agent. Given a multi-hop question, output only the first reasoning step as YAML with three fields: thought, action, target_entity.'}, {'role': 'user', 'content': 'Robinsons and Pocari Sweat are both what kind of product?'}, {'role': 'assistant', 'content': 'thought: "I need to find out what kind of product Robinsons is."\naction: "Lookup"\ntarget_entity: "Robinsons"'}]


In [ ]:
model.eval()
print("dtype:", model.dtype)
print(generate_hop1(train_ds[0]['messages']))

dtype: torch.float32
<class 'transformers.tokenization_utils_base.BatchEncoding'>
thought: "I need to find the nationality of Pearl Lowe first to eventually compare it to Alison Goldfrapp."
action: "Lookup"
target_entity: "Pearl Lowe"


In [ ]:
print(generate_hop1(test_rows[0]['messages']))

<class 'transformers.tokenization_utils_base.BatchEncoding'>
thought: "I need to find out what kind of product Robinsons is categorized as."
action: "Lookup"
target_entity: "Robinsons"


## 7. Run generation over the 95 test questions and save

In [ ]:
import json

with open(f"test_{VARIANT}.jsonl") as f:
    test_rows = [json.loads(line) for line in f]

preds = []
for row in test_rows:
    pred_yaml = generate_hop1(row["messages"])
    preds.append({**row, "pred_yaml": pred_yaml})

out_path = f"preds_{VARIANT}.jsonl"
with open(out_path, "w") as f:
    for p in preds:
        f.write(json.dumps(p) + "\n")

print(f"wrote {len(preds)} predictions to {out_path}")
print("--- sample ---")
print(preds[0]["pred_yaml"])

<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tokenization_utils_base.BatchEncoding'>
<class 'transformers.tok

In [ ]:
# Download the predictions and score them locally through notebook 03.
files.download(out_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>